In [1]:
!pip install torch-geometric scikit-learn matplotlib seaborn pandas numpy

import os
import gc
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, SAGEConv, GCNConv
from torch_geometric.utils import k_hop_subgraph, to_undirected

# Tabular Models (Base & Meta)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import kneighbors_graph

# Metrics & Plotting
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, f1_score,
                             accuracy_score, precision_score, recall_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION
# ============================================================
CONFIG = {
    'data_path': '/kaggle/input/datasets/vagifa/ethereum-frauddetection-dataset/transaction_dataset.csv',
    'save_dir': '/kaggle/working/comparisons/',

    'in_channels':      None,   
    'hidden_channels':  16,        
    'out_channels':     1,

    'epochs':           5,        
    'learning_rate':    0.005,
    'weight_decay':     5e-4,      

    'batch_size':    512,
    'num_hops':      2,
    'knn_k':         5,           

    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)


# ============================================================
# 2. DATA PREPROCESSING & DYNAMIC GRAPH (LEAK-FREE)
# ============================================================
def load_ethereum_data(config):
    print("Loading Ethereum dataset...")
    df = pd.read_csv(config['data_path'])
    df.columns = df.columns.str.strip()
    
    cols_to_drop = ['Index', 'Address', 'ERC20 most sent token type', 'ERC20_most_rec_token_type']
    actual_drop = [c for c in cols_to_drop if c in df.columns]
    
    y = df['FLAG'].values
    X_df = df.drop(columns=actual_drop + ['FLAG'])
    X_df = X_df.apply(pd.to_numeric, errors='coerce')
    
    n_nodes = len(X_df)
    
    # --- 1. SPLIT DATA FIRST TO PREVENT LEAKAGE ---
    # 60% Train (Base Models), 20% Val (Meta-Learner), 20% Test (Evaluation)
    indices = np.arange(n_nodes)
    np.random.seed(42)
    np.random.shuffle(indices)
    
    train_end = int(0.6 * n_nodes)
    val_end   = int(0.8 * n_nodes)
    
    train_idx = indices[:train_end]
    val_idx   = indices[train_end:val_end]
    test_idx  = indices[val_end:]
    
    train_mask = torch.zeros(n_nodes, dtype=torch.bool)
    val_mask   = torch.zeros(n_nodes, dtype=torch.bool)
    test_mask  = torch.zeros(n_nodes, dtype=torch.bool)
    
    train_mask[train_idx] = True
    val_mask[val_idx]     = True
    test_mask[test_idx]   = True

    # --- 2. FIT PREPROCESSORS STRICTLY ON TRAIN DATA ---
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_df.iloc[train_idx]) # Fit ONLY on Train
    X_imputed = imputer.transform(X_df) # Transform all
    
    scaler = StandardScaler()
    scaler.fit(X_imputed[train_idx]) # Fit ONLY on Train
    X_scaled = scaler.transform(X_imputed) # Transform all
    
    # --- 3. BUILD GRAPH ---
    print(f"Building k-NN graph (k={config['knn_k']}) over features...")
    # Note: Building kNN on scaled features is safe transductively as long as stats didn't leak
    A = kneighbors_graph(X_scaled, config['knn_k'], mode='connectivity', include_self=False, n_jobs=-1)
    coo = A.tocoo()
    edge_index = torch.tensor(np.vstack((coo.row, coo.col)), dtype=torch.long)
    edge_index = to_undirected(edge_index)
    
    x = torch.tensor(X_scaled, dtype=torch.float)
    y = torch.tensor(y, dtype=torch.long)
    
    data = Data(x=x, edge_index=edge_index, y=y)
    data.train_mask = train_mask
    data.val_mask   = val_mask
    data.test_mask  = test_mask
    
    config['in_channels'] = x.shape[1]
    
    print(f"Data Loaded: {data.num_nodes} nodes, {data.num_edges} edges.")
    print(f"Splits - Train: {train_mask.sum()}, Val: {val_mask.sum()}, Test: {test_mask.sum()}")
    return data


# ============================================================
# 3. GRAPH SAMPLER & MODELS
# ============================================================
class SubgraphBatchSampler:
    def __init__(self, node_indices, batch_size, num_hops, edge_index, num_nodes, shuffle=True):
        self.node_indices = node_indices
        self.batch_size   = batch_size
        self.num_hops     = num_hops
        self.edge_index   = edge_index
        self.num_nodes    = num_nodes
        self.shuffle      = shuffle

    def __iter__(self):
        idx = self.node_indices.clone()
        if self.shuffle:
            idx = idx[torch.randperm(len(idx))]

        for start in range(0, len(idx), self.batch_size):
            seeds = idx[start: start + self.batch_size]
            subset, sub_edge_index, mapping, _ = k_hop_subgraph(
                node_idx=seeds, num_hops=self.num_hops, edge_index=self.edge_index,
                relabel_nodes=True, num_nodes=self.num_nodes, flow='source_to_target'
            )
            yield subset, sub_edge_index, mapping

    def __len__(self):
        return (len(self.node_indices) + self.batch_size - 1) // self.batch_size


class BaselineGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, gnn_type='GCN'):
        super().__init__()
        if gnn_type == 'GCN':
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, hidden_channels)
        elif gnn_type == 'SAGE':
            self.conv1 = SAGEConv(in_channels, hidden_channels)
            self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        elif gnn_type == 'GAT':
            self.conv1 = GATConv(in_channels, hidden_channels, heads=1)
            self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1)
            
        self.classifier = Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.classifier(x)


# ============================================================
# 4. TRAINING & EVALUATION ROUTINES
# ============================================================
def get_graph_predictions(model, data, sampler, config):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for subset, sub_ei, mapping in sampler:
            out = model(data.x[subset].to(config['device']), sub_ei.to(config['device'])).squeeze()
            if out.dim() == 0: out = out.unsqueeze(0)
            out_seed = out[mapping]
            all_logits.append(out_seed.cpu())
            
    logits = torch.cat(all_logits)
    probs  = torch.sigmoid(logits).numpy()
    return probs

def train_graph_model(model, data, config, model_name="Model"):
    print(f"\n--- Training Graph Model: {model_name} ---")
    train_idx = torch.where(data.train_mask)[0]
    val_idx   = torch.where(data.val_mask)[0]
    test_idx  = torch.where(data.test_mask)[0]

    train_sampler = SubgraphBatchSampler(train_idx, config['batch_size'], config['num_hops'], data.edge_index, data.num_nodes, shuffle=True)
    
    # Un-shuffled samplers to align indices correctly for stacking evaluation
    val_sampler  = SubgraphBatchSampler(val_idx, config['batch_size'] * 2, config['num_hops'], data.edge_index, data.num_nodes, shuffle=False)
    test_sampler = SubgraphBatchSampler(test_idx, config['batch_size'] * 2, config['num_hops'], data.edge_index, data.num_nodes, shuffle=False)

    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])

    for epoch in range(1, config['epochs'] + 1):
        model.train()
        for subset, sub_ei, mapping in train_sampler:
            optimizer.zero_grad()
            out = model(data.x[subset].to(config['device']), sub_ei.to(config['device'])).squeeze()
            if out.dim() == 0: out = out.unsqueeze(0)
            out_seed = out[mapping]
            y_seed = data.y[subset][mapping].float().to(config['device'])
            
            loss = criterion(out_seed, y_seed)
            loss.backward()
            optimizer.step()
    
    # Return Val (for Meta Learner training) and Test (for Final Eval)
    val_probs  = get_graph_predictions(model, data, val_sampler, config)
    test_probs = get_graph_predictions(model, data, test_sampler, config)
    
    return val_probs, test_probs


def run_tabular_baselines(data):
    print("\n--- Training Tabular ML Baselines ---")
    train_idx = data.train_mask.nonzero(as_tuple=True)[0]
    val_idx   = data.val_mask.nonzero(as_tuple=True)[0]
    test_idx  = data.test_mask.nonzero(as_tuple=True)[0]

    X_train = data.x[train_idx].numpy()
    y_train = data.y[train_idx].numpy()
    X_val   = data.x[val_idx].numpy()
    X_test  = data.x[test_idx].numpy()
    
    ml_models = {
        'LR': LogisticRegression(max_iter=100, solver='liblinear'),
        'SVM': SVC(probability=True, max_iter=250),
        'MLP': MLPClassifier(hidden_layer_sizes=(16,), max_iter=50, random_state=42)
    }

    base_probs = {}
    for name, clf in ml_models.items():
        print(f"Training {name}...")
        clf.fit(X_train, y_train)
        
        val_probs  = clf.predict_proba(X_val)[:, 1]
        test_probs = clf.predict_proba(X_test)[:, 1]
        
        base_probs[name] = {'val': val_probs, 'test': test_probs}
        
    return base_probs


def train_meta_learner(stack_name, models_to_stack, base_model_probs, y_val):
    print(f"Training Meta Learner for: {stack_name} ...")
    
    # Construct Meta-Features using predictions from the Validation Set (Prevents Overfitting/Leakage)
    X_meta_train = np.column_stack([base_model_probs[m]['val'] for m in models_to_stack])
    X_meta_test  = np.column_stack([base_model_probs[m]['test'] for m in models_to_stack])
    
    meta_lr = LogisticRegression(max_iter=100, solver='liblinear')
    meta_lr.fit(X_meta_train, y_val)
    
    meta_test_probs = meta_lr.predict_proba(X_meta_test)[:, 1]
    meta_test_preds = meta_lr.predict(X_meta_test)
    
    return meta_test_preds, meta_test_probs


# ============================================================
# 5. COMPARISON PIPELINE & PLOTTING
# ============================================================
def plot_combined_roc(results, config):
    plt.figure(figsize=(14, 12))
    colors = sns.color_palette("tab20", len(results))
    
    for (name, (y_true, preds, probs)), color in zip(results.items(), colors):
        fpr, tpr, _ = roc_curve(y_true, probs)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.4f})')

    plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--')
    plt.title('Combined ROC Curve Comparison', fontsize=16, weight='bold')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(config['save_dir'], '1_Combined_ROC.png'), dpi=300)
    plt.close()


def plot_combined_pr(results, config):
    plt.figure(figsize=(14, 12))
    colors = sns.color_palette("tab20", len(results))
    
    for (name, (y_true, preds, probs)), color in zip(results.items(), colors):
        precision, recall, _ = precision_recall_curve(y_true, probs)
        pr_auc = auc(recall, precision)
        plt.plot(recall, precision, color=color, lw=2, label=f'{name} (PR AUC = {pr_auc:.4f})')

    plt.title('Combined Precision-Recall Curve Comparison', fontsize=16, weight='bold')
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.legend(loc="lower left", fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(config['save_dir'], '2_Combined_PR.png'), dpi=300)
    plt.close()


def plot_confusion_matrices_grid(results, config):
    num_models = len(results)
    cols = 4
    rows = math.ceil(num_models / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4.5))
    axes = axes.flatten()

    for idx, (name, (y_true, preds, probs)) in enumerate(results.items()):
        cm = confusion_matrix(y_true, preds)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 10}, ax=axes[idx])
        axes[idx].set_title(name, fontsize=11, weight='bold')
        axes[idx].set_xticklabels(['Valid (0)', 'Fraud (1)'])
        axes[idx].set_yticklabels(['Valid (0)', 'Fraud (1)'])
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')

    for empty_idx in range(num_models, len(axes)):
        fig.delaxes(axes[empty_idx])

    plt.tight_layout()
    plt.savefig(os.path.join(config['save_dir'], '3_Confusion_Matrices_Grid.png'), dpi=300)
    plt.close()


def generate_comparison_table_and_reports(results, config):
    print("\n" + "="*80)
    print("CLASSIFICATION REPORTS (4 DIGITS) FOR ALL MODELS & STACKS")
    print("="*80)
    
    metrics_data = []
    
    for name, (y_true, preds, probs) in results.items():
        print(f"\n[{name}] Classification Report:")
        print(classification_report(y_true, preds, digits=4, target_names=['Valid (0)', 'Fraud (1)'], zero_division=0))
        
        acc = accuracy_score(y_true, preds)
        mac_p = precision_score(y_true, preds, average='macro', zero_division=0)
        mac_r = recall_score(y_true, preds, average='macro', zero_division=0)
        mac_f1 = f1_score(y_true, preds, average='macro', zero_division=0)
        
        metrics_data.append([name, acc, mac_p, mac_r, mac_f1])
        
    df_metrics = pd.DataFrame(metrics_data, columns=['Model', 'Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1'])
    
    print("\n" + "="*80)
    print("SUMMARY METRICS TABLE (MACRO AVERAGES)")
    print("="*80)
    print(df_metrics.to_string(index=False, float_format="{:.6f}".format))
    
    df_metrics.to_csv(os.path.join(config['save_dir'], '4_Comparison_Metrics.csv'), index=False)
    print(f"\nAll comparison outputs saved to: {config['save_dir']}")


# ============================================================
# 6. MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    
    # 1. Load Data
    data = load_ethereum_data(CONFIG)
    
    y_val  = data.y[data.val_mask].numpy()
    y_test = data.y[data.test_mask].numpy()
    
    all_results = {}
    base_model_probs = {}
    
    # 2. Run Tabular Baselines
    tab_probs_dict = run_tabular_baselines(data)
    
    for name, probs in tab_probs_dict.items():
        base_model_probs[name] = probs
        test_preds = (probs['test'] > 0.5).astype(int)
        all_results[name] = (y_test, test_preds, probs['test'])
        
    # 3. Run Graph Baselines
    graph_baselines = ['GCN', 'SAGE', 'GAT']
    for gnn_type in graph_baselines:
        model = BaselineGNN(CONFIG['in_channels'], CONFIG['hidden_channels'], 
                            CONFIG['out_channels'], gnn_type=gnn_type).to(CONFIG['device'])
        
        display_name = "GNN (GraphSAGE)" if gnn_type == 'SAGE' else f"GNN ({gnn_type})"
        val_probs, test_probs = train_graph_model(model, data, CONFIG, model_name=display_name)
        
        base_model_probs[display_name] = {'val': val_probs, 'test': test_probs}
        test_preds = (test_probs > 0.5).astype(int)
        all_results[display_name] = (y_test, test_preds, test_probs)


    # 4. Run Stacking Combinations
    print("\n--- Training Stacking Ensembles ---")
    
    stacking_combinations = {
        'Stack (LR + MLP)': ['LR', 'MLP'],
        'Stack (LR + GNN-GraphSAGE)': ['LR', 'GNN (GraphSAGE)'],
        'Stack (LR + GNN-GCN)': ['LR', 'GNN (GCN)'],
        'Stack (MLP + GNN-GraphSAGE)': ['MLP', 'GNN (GraphSAGE)'],
        'Stack (MLP + GNN-GCN)': ['MLP', 'GNN (GCN)'],
        'Stack (GNN-GraphSAGE + GNN-GCN)': ['GNN (GraphSAGE)', 'GNN (GCN)'],
        'Stack (LR + MLP + GNN-GraphSAGE)': ['LR', 'MLP', 'GNN (GraphSAGE)'],
        'Stack (LR + MLP + GNN-GCN)': ['LR', 'MLP', 'GNN (GCN)'],
        'Stack (LR + GNN-GraphSAGE + GNN-GCN)': ['LR', 'GNN (GraphSAGE)', 'GNN (GCN)'],
        'Stack (MLP + GNN-GraphSAGE + GNN-GCN)': ['MLP', 'GNN (GraphSAGE)', 'GNN (GCN)'],
        'Stack (LR + MLP + GNN-GraphSAGE + GNN-GCN)': ['LR', 'MLP', 'GNN (GraphSAGE)', 'GNN (GCN)']
    }

    for stack_name, models in stacking_combinations.items():
        meta_preds, meta_probs = train_meta_learner(stack_name, models, base_model_probs, y_val)
        all_results[stack_name] = (y_test, meta_preds, meta_probs)


    # 5. Generate Visual Comparisons & Metrics
    print("\nGenerating final comparison charts...")
    plot_combined_roc(all_results, CONFIG)
    plot_combined_pr(all_results, CONFIG)
    plot_confusion_matrices_grid(all_results, CONFIG)
    generate_comparison_table_and_reports(all_results, CONFIG)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.4 MB/s eta 0:00:00
Loading Ethereum dataset...
Building k-NN graph (k=5) over features...
Data Loaded: 9841 nodes, 68044 edges.
Splits - Train: 5904, Val: 1968, Test: 1969

--- Training Tabular ML Baselines ---
Training LR...
Training SVM...
Training MLP...

--- Training Graph Model: GNN (GCN) ---

--- Training Graph Model: GNN (GraphSAGE) ---

--- Training Graph Model: GNN (GAT) ---

--- Training Stacking Ensembles ---
Training Meta Learner for: Stack (LR + MLP) ...
Training Meta Learner for: Stack (LR + GNN-GraphSAGE) ...
Training Meta Learner for: Stack (LR + GNN-GCN) ...
Training Meta Learner for: Stack (MLP + GNN-GraphSAGE) ...
Training Meta Learner for: Stack (MLP + GNN-GCN) ...
Training Meta Learner for: Stack (GNN-GraphSAGE + GNN-GCN) ...
Training Meta Learner for: Stack (LR + MLP + GNN-GraphSAGE) ...
Training Meta Learner for: Stack (LR + ML